# Problema Gravitacional de N-Cuerpos y Estabilidad Orbital

**Presentado por:** Eduardo Pérez Cárdenas

## Fase I: Fundamentos Matemáticos y Diferenciación Numérica

Para modelar la evolución de un sistema de N-Cuerpos, la dinámica dicta que la aceleración de cualquier cuerpo $i$ es la suma vectorial de los gradientes gravitacionales producidos por todos los demás cuerpos $j$:
$$ \frac{d^2\vec{r}_i}{dt^2} = - \sum_{j \neq i}^{N} G m_j \frac{\vec{r}_i - \vec{r}_j}{|\vec{r}_i - \vec{r}_j|^3} $$

### Respuesta Pregunta 1: Deducción de la Diferenciación de Lagrange (Ec. 0.3)
Para aproximar estas derivadas computacionalmente, partimos del polinomio de interpolación de Lagrange:
$$ f(x) \approx \sum_{j=0}^{n} l_j(x) f_j \quad \text{donde} \quad l_j(x) = \frac{\pi(x)}{(x - x_j)\pi'(x_j)} $$

Definimos $\pi(x) = \prod_{i=0}^n (x - x_i)$. Derivando la aproximación polinómica respecto a $x$, obtenemos:
$$ f'(x) \approx \sum_{j=0}^{n} l'_j(x) f_j $$

Aplicando la regla del cociente, la derivada del polinomio base $l_j(x)$ es:
$$ l'_j(x) = \frac{\pi'(x)(x - x_j) - \pi(x)}{(x - x_j)^2 \pi'(x_j)} $$

Para una malla de puntos uniformemente espaciados con un tamaño de paso $h$, introducimos el cambio de variable $x = x_0 + sh$. Esto implica que $dx = h \, ds$ y, por la regla de la cadena, el operador diferencial se transforma en $\frac{d}{dx} = \frac{1}{h}\frac{d}{ds}$.

La distancia entre los nodos espaciados de la malla se expresa como $x_j - x_i = (j - i)h$. Al evaluar analíticamente el denominador $\pi'(x_j)$, obtenemos la constante discreta:
$$ \pi'(x_j) = \prod_{i=0, i \neq j}^{n} (x_j - x_i) = h^n (-1)^{n-j} j! (n-j)! $$

Para el numerador, expresamos el término $\frac{\pi(x)}{x-x_j}$ como un polinomio en la variable paramétrica $s$ con coeficientes $a_{jk}$:
$$ \frac{\pi(x)}{x-x_j} = h^n \sum_{k=1}^{n+1} a_{jk} s^{k-1} $$

Al aplicar el operador derivada $\frac{d}{ds}$ a este polinomio, el término se transforma en $\sum_{k=1}^{n} k a_{jk} s^{k-1}$.

Finalmente, sustituyendo la constante $\pi'(x_j)$ y la derivada en $s$ en la sumatoria original, y factorizando el término $\frac{1}{h}$ proveniente de la regla de la cadena, se obtiene directamente la Fórmula General de Diferenciación Numérica de $n$ puntos:
$$ f'(x) = \frac{1}{h} \left[ \frac{(-1)^n}{n! a_{0n}} \left( \sum_{k=1}^n k a_{0k} s^{k-1} \right) f_0 + \sum_{j=1}^n \left\{ \frac{(-1)^{n-j}}{j!(n-j)! a_{jn}} \left( \sum_{k=1}^n k a_{jk} s^{k-1} \right) \right\} f_j \right] $$

### Respuesta Pregunta 2: El caso especial de Diferencia Central (s=1)
Se requiere demostrar que la diferencia central clásica es un caso especial de la fórmula general de primera derivada de 3 puntos (Ec. 0.4) cuando se evalúa en el nodo central, es decir, fijando el parámetro $s=1$.

La fórmula general para $n=2$ está dada por:
$$ f'(x) = \frac{(-3+2s)f_0 + 4(1-s)f_1 + (-1+2s)f_2}{2h} $$

Sustituyendo $s=1$:
$$ f'(x) = \frac{(-3+2(1))f_0 + 4(1-1)f_1 + (-1+2(1))f_2}{2h} $$

Al resolver los coeficientes algebraicos obtenemos:
- Coeficiente de $f_0$: $-3 + 2 = -1$
- Coeficiente de $f_1$: $4(0) = 0$
- Coeficiente de $f_2$: $-1 + 2 = 1$

Por lo tanto, la expresión se reduce de forma estricta a la fórmula canónica de diferencia central:
$$ f'(x) = \frac{-f_0 + 0 + f_2}{2h} = \frac{f_2 - f_0}{2h} \quad \blacksquare $$

In [2]:
# Fase II: Implementación Algorítmica (Python Puro, sin librerías de EDOs externas)
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings("ignore")

# Sistema normalizado: Masas Solares, Unidades Astronómicas (AU), y Años.
G = 4.0 * math.pi**2

In [3]:
def aceleracion_3d(masas, posiciones):
    """Calcula el vector de aceleración [ax, ay, az] para N cuerpos."""
    N = len(masas)
    aceleraciones = [[0.0, 0.0, 0.0] for _ in range(N)]
    
    for i in range(N):
        ax, ay, az = 0.0, 0.0, 0.0
        for j in range(N):
            if i != j:
                dx = posiciones[j][0] - posiciones[i][0]
                dy = posiciones[j][1] - posiciones[i][1]
                dz = posiciones[j][2] - posiciones[i][2]
                dist_sq = dx**2 + dy**2 + dz**2
                dist = math.sqrt(dist_sq)
                
                # Suavizado (softening)
                epsilon = 1e-9 
                fuerza_mag = (G * masas[j]) / (dist_sq * dist + epsilon)
                
                ax += fuerza_mag * dx
                ay += fuerza_mag * dy
                az += fuerza_mag * dz
        aceleraciones[i] = [ax, ay, az]
    return aceleraciones

# ---- SOLVER 1: RUNGE-KUTTA DE 4TO ORDEN (Alta precisión local) ----
def paso_rk4(masas, pos, vel, dt):
    """Integra un paso de tiempo dt usando RK4 para vectores 3D."""
    N = len(masas)
    
    def suma_vectores(v1, v2, c):
        return [v1[k] + c * v2[k] for k in range(3)]

    # k1
    a1 = aceleracion_3d(masas, pos)
    k1_v, k1_r = a1, vel
    
    # k2
    pos2 = [suma_vectores(pos[i], k1_r[i], dt/2.0) for i in range(N)]
    vel2 = [suma_vectores(vel[i], k1_v[i], dt/2.0) for i in range(N)]
    a2 = aceleracion_3d(masas, pos2)
    k2_v, k2_r = a2, vel2
    
    # k3
    pos3 = [suma_vectores(pos[i], k2_r[i], dt/2.0) for i in range(N)]
    vel3 = [suma_vectores(vel[i], k2_v[i], dt/2.0) for i in range(N)]
    a3 = aceleracion_3d(masas, pos3)
    k3_v, k3_r = a3, vel3
    
    # k4
    pos4 = [suma_vectores(pos[i], k3_r[i], dt) for i in range(N)]
    vel4 = [suma_vectores(vel[i], k3_v[i], dt) for i in range(N)]
    a4 = aceleracion_3d(masas, pos4)
    k4_v, k4_r = a4, vel4
    
    pos_new, vel_new = [], []
    for i in range(N):
        p_n = [pos[i][j] + (dt/6.0)*(k1_r[i][j] + 2*k2_r[i][j] + 2*k3_r[i][j] + k4_r[i][j]) for j in range(3)]
        v_n = [vel[i][j] + (dt/6.0)*(k1_v[i][j] + 2*k2_v[i][j] + 2*k3_v[i][j] + k4_v[i][j]) for j in range(3)]
        pos_new.append(p_n)
        vel_new.append(v_n)
        
    return pos_new, vel_new

# ---- SOLVER 2: INTEGRADOR SIMPLÉCTICO (Conservación de Energía a largo plazo) ----
def paso_simplectico(masas, pos, vel, dt):
    """Integra usando Euler Semi-Implícito (Método Simpléctico de 1er orden)."""
    N = len(masas)
    acel = aceleracion_3d(masas, pos)
    
    vel_new, pos_new = [], []
    for i in range(N):
        v_n = [vel[i][j] + acel[i][j] * dt for j in range(3)]
        p_n = [pos[i][j] + v_n[j] * dt for j in range(3)]
        vel_new.append(v_n)
        pos_new.append(p_n)
        
    return pos_new, vel_new

# ---- DETECCIÓN DE EVENTOS: BISECCIÓN ----
def biseccion_evento(f, a, b, tol=1e-7):
    """Encuentra la raíz de una función f(x) en el intervalo [a, b]."""
    if f(a) * f(b) >= 0:
        return None # No hay cruce de raíz en este intervalo
    
    medio = a
    while (b - a) >= tol:
        medio = (a + b) / 2.0
        if f(medio) == 0.0:
            break
        if f(medio) * f(a) < 0:
            b = medio
        else:
            a = medio
    return medio

In [4]:
# ### Respuesta Pregunta 3: Validación del Método de Bisección
# El documento exige verificar la lógica de búsqueda de raíces para asegurar 
# que puede identificar la constante de cruce "2.667" en un caso de prueba normalizado.

# Definimos una función de prueba f(x) cuya raíz analítica es exactamente 8/3 (2.6666...)
def funcion_prueba(x):
    return 3 * x - 8

# Ejecutamos nuestra función de bisección en el intervalo [0, 5]
raiz_detectada = biseccion_evento(funcion_prueba, 0, 5, tol=1e-6)

print("--- VALIDACIÓN DE DETECCIÓN DE EVENTOS (BISECCIÓN) ---")
print(f"Intervalo de búsqueda: [0, 5]")
print(f"Raíz detectada numéricamente: {raiz_detectada:.3f}")

if round(raiz_detectada, 3) == 2.667:
    print("ESTADO: ÉXITO. El solver de bisección ha identificado la constante de cruce correctamente.")
else:
    print("ESTADO: FALLO en la validación.")

--- VALIDACIÓN DE DETECCIÓN DE EVENTOS (BISECCIÓN) ---
Intervalo de búsqueda: [0, 5]
Raíz detectada numéricamente: 2.667
ESTADO: ÉXITO. El solver de bisección ha identificado la constante de cruce correctamente.


## 2. Fase III: Benchmark Analítico a 2 Cuerpos y Análisis de Error (Task 1)

Para validar la fidelidad de nuestros integradores, el proyecto exige modelar un satélite real frente a la Tierra y medir el "Solver Drift" (la deriva del integrador) tras 100 periodos orbitales.

### Datos Astronómicos Externos (ISS)
Utilizaremos las condiciones reales de la Estación Espacial Internacional (ISS) proyectadas en nuestro sistema normalizado (Masas Solares, Unidades Astronómicas y Años).
* **Masa de la Tierra ($M_{\oplus}$):** $3.003 \times 10^{-6} M_{\odot}$
* **Semieje mayor ISS ($a$):** $6812.137 \text{ km} \approx 4.5536 \times 10^{-5} \text{ UA}$
* **Excentricidad ISS ($e$):** $0.000681$
* **Inclinación:** Omitida temporalmente en este benchmark 2D para aislar el error de truncamiento radial.

> *Fuente empírica: Parámetros orbitales extraídos del catálogo TLE de CelesTrak (https://celestrak.org/).*

### El "Solver Drift" y el Error Teórico
En una órbita kepleriana ideal de 2 cuerpos, tras exactamente 100 órbitas, el satélite debe regresar matemáticamente a sus coordenadas iniciales $(x_0, y_0, z_0)$. Cualquier diferencia entre la posición numérica final y la inicial es el **Solver Drift**.

El documento proporciona la cota del error de truncamiento teórico de la primera derivada para un esquema polinómico de grado 3 ($n=3$):
$$ E'_3 = -\frac{h^3}{12}(3 - 11s + 9s^2 - 2s^3)f^{iv}(\xi) $$

Nuestro objetivo es medir el Drift del integrador de Runge-Kutta 4 (RK4) y del integrador Simpléctico, compararlo con esta cota $\mathcal{O}(h^3)$, y observar cómo conservan la energía del sistema con el paso del tiempo.